# β stabilisation and mode type: GFTM (FILTER=0.5) and TGLF against GS2 (M1_A1_NE, R3, R4)

γ vs β at fixed ky, one figure per code against GS2, for each case. Codes: GFTM widths 0.4 / 0.6 / 0.7 / 0.8
(`scan_beta_f05`, FILTER=0.5), TGLF default / V5 (`scan_beta_modes`) and TGLF V3 (older `scan_beta` scan). All scans
use GS2's own (ky, β) grid; TGLF V3's M1_A1_NE scan has two extra ky (0.2, 0.3) that are dropped. The single-ky
figures draw all three TGLF settings next to each code.

**Line styles.** GS2 black (dominant mode only). Code: dominant mode (`argmax` of `growth_rate` over `mode`, never
index 0) solid, the other mode dashed (`NMODES=2`).

**Markers = mode type** from `general_analysis.mode_classification` (cutoffs `T_BAL`, `T_TEAR`, see that module):
▲ MTM (tearing parity), ■ KBM (ballooning parity), ● mixed. Markers are drawn only where γ > 0.
- **GS2** uses all four indicators (tearing parameter T, ω sign, χe/χi, De/χe) from each run's final time.
- **GFTM / TGLF** linear runs have no per-mode flux weights, so they are classified on **T and ω only**, with the
  same cutoffs: KBM = ω>0 and T<`T_BAL`; MTM = ω<0 and T>`T_TEAR`; otherwise mixed. A code marker is therefore a
  looser test than a GS2 one, and "KBM" there means only "ballooning parity, ion direction".
- T for GFTM / TGLF is the `tearing_parameter` stored per mode in each leaf's `pyroscan.nc`. TGLF V3's older scan
  stores none, so its T is computed by `FieldLine` from the `A_par` eigenfunction in its `pyroscan.nc` (`eig_T`).
- Every scan is read with `PyroScan(pyroscan_json=..., load_base_pyro=True).load_gk_output(netcdf_file=...)`;
  no per-run outputs are loaded, except GS2's γ(t) in section (4), which a `pyroscan.nc` does not store.

**β axis.** Code scans store pyrokinetics β (= GS2's number × a per-case factor f); the x axis here is **GS2's β
number**, matched by index. f is printed below and checked constant. TGLF V3 stores GS2's number directly (f = 1),
but its `input.tglf` BETAE equals TGLF V5's at the same index (checked R4 and M1_A1_NE, lowest and highest β), so
the index match is the same physical β.

## Imports and settings

In [1]:
from pathlib import Path
import os
import numpy as np
import xarray as xr
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pyrokinetics import Pyro, PyroScan
from pyrokinetics.diagnostics.field_line import FieldLine
from general_analysis import mode_classification as mc

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "beta_stabilisation_f05"
data_root = Path(os.environ["GYRO_DATA_OUTPUT"]).expanduser()
run_template = "Runs"

cases = ["M1_A1_NE", "R3", "R4"]
reference_beta = {"M1_A1_NE": 0.036, "R3": 0.094, "R4": 0.053}   # same beta convention as the plot axis
ky_targets = [0.1, 0.2, 0.3, 0.5]          # snapped to the nearest GS2 grid value, shown in the panel titles

# GS2 reference: data_root / code / run_template / project / case / scan_information
gs2_code, gs2_project, gs2_scan = "GS2", "scan_beta", "parameter_scan_GS2"
# Codes, one figure each: label -> (code, project, scan_information = leaf). Same three path parts for every case.
codes = {
    "GFTM w0.4": ("GFTM", "scan_beta_f05", "w0p4"),
    "GFTM w0.6": ("GFTM", "scan_beta_f05", "w0p6"),
    "GFTM w0.7": ("GFTM", "scan_beta_f05", "w0p7"),
    "GFTM w0.8": ("GFTM", "scan_beta_f05", "w0p8"),
    "TGLF default": ("TGLF", "scan_beta_modes", "default"),
    "TGLF V5": ("TGLF", "scan_beta_modes", "v5"),
    "TGLF V3": ("TGLF", "scan_beta", "parameter_scan_tglf_v3"),   # older scan: no tearing_parameter, beta stored as GS2's number
}
metadata_file, output_file = "pyroscan.json", "pyroscan.nc"   # in every scan directory
gamma_floor = 0.0                                        # markers only where gamma > this
n_spot, seed = 3, 0                                      # spot-check cells per (case, code family)

## Load data

Every leaf as a `PyroScan` from its `pyroscan.json` + `pyroscan.nc` (base pyro from `pyroscan_base.input`); the
plain datasets below are its `gk_output` with units stripped. GS2 and the codes must share the ky grid exactly; nothing is
interpolated.

In [2]:
def scan(code, project, case, leaf):
    d = data_root / code / run_template / project / case / leaf
    s = PyroScan(pyroscan_json=d / metadata_file, load_base_pyro=True)
    s.load_gk_output(netcdf_file=d / output_file)
    return s

gs2_pyroscan = {c: scan(gs2_code, gs2_project, c, gs2_scan) for c in cases}
model_pyroscan = {(c, lab): scan(code, project, c, leaf) for c in cases for lab, (code, project, leaf) in codes.items()}
gs2 = {c: s.gk_output.data.pint.dequantify() for c, s in gs2_pyroscan.items()}
model = {k: s.gk_output.data.pint.dequantify() for k, s in model_pyroscan.items()}

for (c, lab), ds in model.items():   # exact selection on GS2's ky (KeyError if one is missing); report what is dropped
    extra = np.setdiff1d(ds.ky.values, gs2[c].ky.values)
    if extra.size:
        print(c, lab, "ky not in GS2, dropped:", extra)
        model[(c, lab)] = ds.sel(ky=gs2[c].ky.values)

for c in cases:
    for lab in codes:
        ds = model[(c, lab)]
        assert np.array_equal(ds.ky.values, gs2[c].ky.values), (c, lab, "ky grid differs from GS2")
        ratio = ds.beta.values / gs2[c].beta.values
        assert np.allclose(ratio, ratio[0], rtol=1e-3), (c, lab, "beta is not GS2 beta x constant")
    print(c, "ky", gs2[c].ky.size, "beta", gs2[c].beta.size, "| pyro beta / GS2 beta f =",
          {lab: round(float(model[(c, lab)].beta.values[0] / gs2[c].beta.values[0]), 4) for lab in codes},
          "| modes", {lab: int(model[(c, lab)].mode.size) for lab in codes})

/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0000'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0003'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0004'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0005'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0006'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0007'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0008'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0009'. 
  warnings.warn(


M1_A1_NE TGLF V3 ky not in GS2, dropped: [0.2 0.3]
M1_A1_NE ky 40 beta 19 | pyro beta / GS2 beta f = {'GFTM w0.4': 1.0, 'GFTM w0.6': 1.0, 'GFTM w0.7': 1.0, 'GFTM w0.8': 1.0, 'TGLF default': 1.0, 'TGLF V5': 1.0, 'TGLF V3': 1.0} | modes {'GFTM w0.4': 2, 'GFTM w0.6': 2, 'GFTM w0.7': 2, 'GFTM w0.8': 2, 'TGLF default': 2, 'TGLF V5': 2, 'TGLF V3': 2}
R3 ky 42 beta 19 | pyro beta / GS2 beta f = {'GFTM w0.4': 0.9546, 'GFTM w0.6': 0.9546, 'GFTM w0.7': 0.9546, 'GFTM w0.8': 0.9546, 'TGLF default': 0.9546, 'TGLF V5': 0.9546, 'TGLF V3': 1.0} | modes {'GFTM w0.4': 2, 'GFTM w0.6': 2, 'GFTM w0.7': 2, 'GFTM w0.8': 2, 'TGLF default': 2, 'TGLF V5': 2, 'TGLF V3': 2}
R4 ky 42 beta 19 | pyro beta / GS2 beta f = {'GFTM w0.4': 0.8776, 'GFTM w0.6': 0.8776, 'GFTM w0.7': 0.8776, 'GFTM w0.8': 0.8776, 'TGLF default': 0.8776, 'TGLF V5': 0.8776, 'TGLF V3': 1.0} | modes {'GFTM w0.4': 2, 'GFTM w0.6': 2, 'GFTM w0.7': 2, 'GFTM w0.8': 2, 'TGLF default': 2, 'TGLF V5': 2, 'TGLF V3': 2}


In [3]:
_gs2_ind = {}
def gs2_indicators(case, i, j):
    "mode_classification.indicators on one (ky, beta) cell of GS2's pyroscan.nc"
    if case not in _gs2_ind:   # whole scan once: mc.indicators(scan, ds) takes a scan and its samples
        s = mc.attach_legacy_funcs(gs2_pyroscan[case])
        ind = mc.indicators(s, mc.as_samples(s, s.gk_output.data))
        _gs2_ind[case] = {n: ind[n].values.reshape(s.gk_output.data.growth_rate.shape) for n in ("T", "E", "omega", "chi_ratio")}
    return {n: float(v[i, j]) for n, v in _gs2_ind[case].items()}

## Calculate and select

Nearest GS2 ky to each target (actual values printed). GS2 type per plotted (ky, β) cell from GS2's `pyroscan.nc`
through `mode_classification.indicators` + `classify`: the scan varies only ky and β (`p_prime_type` 0), so the base
pyro's species and geometry hold for every cell. ω there is the scan's time average over the last 20%, not the final
time (differences < 0.01 on checked cells; T, χe/χi, De/χe identical). Code type from stored `tearing_parameter` and
`mode_frequency` per mode with the same cutoffs (T and ω only, see header).

In [5]:
ky_index = {c: [int(np.abs(gs2[c].ky.values - k).argmin()) for k in ky_targets] for c in cases}
for c in cases:
    print(c, "ky target -> GS2 grid:", {k: round(float(gs2[c].ky.values[i]), 4) for k, i in zip(ky_targets, ky_index[c])})

_gs2_ind = {}
def gs2_indicators(case, i, j):
    "mode_classification.indicators on one (ky, beta) cell of GS2's pyroscan.nc"
    if case not in _gs2_ind:   # whole scan once: mc.indicators(scan, ds) takes a scan and its samples
        s = mc.attach_legacy_funcs(gs2_pyroscan[case])
        ind = mc.indicators(s, mc.as_samples(s, s.gk_output.data))
        _gs2_ind[case] = {n: ind[n].values.reshape(s.gk_output.data.growth_rate.shape) for n in ("T", "E", "omega", "chi_ratio")}
    return {n: float(v[i, j]) for n, v in _gs2_ind[case].items()}

gs2_label = {(c, i, j): str(mc.classify(gs2_indicators(c, i, j)))
             for c in cases for i in ky_index[c] for j in range(gs2[c].beta.size)
             if gs2[c].growth_rate.values[i, j] > gamma_floor}

def code_type(T, omega):
    "same cutoffs as mode_classification.classify, on the indicators a GFTM/TGLF linear run has"
    w = mc.ION_DIRECTION * omega
    return np.where((w > 0) & (T < mc.T_BAL), "KBM", np.where((w < 0) & (T > mc.T_TEAR), "MTM", "mixed"))

def eig_T(pyroscan, ky=None):
    "FieldLine T of every (ky, beta, mode) from the scan's stored A_par eigenfunction, optionally on a subset of ky"
    p = pyroscan.base_pyro
    p.gk_output = xr.Dataset({"apar": pyroscan.gk_output.data.eigenfunctions.sel(field="apar")})
    T = FieldLine(p).compute_linear_tearing_parameter().transpose("ky", "beta", "mode")
    return (T if ky is None else T.sel(ky=ky)).values

for k, ds in model.items():   # TGLF V3's older scan stores no tearing_parameter: take it from its stored A_par
    if "tearing_parameter" not in ds:
        model[k] = ds.assign(tearing_parameter=(ds.growth_rate.dims, eig_T(model_pyroscan[k], ds.ky.values)))
code_label = {k: code_type(ds.tearing_parameter.values, ds.mode_frequency.values) for k, ds in model.items()}
dominant_mode = {k: ds.growth_rate.fillna(-np.inf).argmax("mode").values for k, ds in model.items()}   # (ky, beta)
for c in cases:
    n = {t: sum(1 for k, v in gs2_label.items() if k[0] == c and v == t) for t in ("KBM", "MTM", "mixed")}
    print(c, "GS2 type counts at plotted ky (gamma>0):", n)

M1_A1_NE ky target -> GS2 grid: {0.1: 0.1115, 0.2: 0.1877, 0.3: 0.2892, 0.5: 0.4923}
R3 ky target -> GS2 grid: {0.1: 0.1115, 0.2: 0.2, 0.3: 0.3, 0.5: 0.4923}
R4 ky target -> GS2 grid: {0.1: 0.1115, 0.2: 0.2, 0.3: 0.3, 0.5: 0.4923}


M1_A1_NE GS2 type counts at plotted ky (gamma>0): {'KBM': 43, 'MTM': 4, 'mixed': 2}
R3 GS2 type counts at plotted ky (gamma>0): {'KBM': 32, 'MTM': 31, 'mixed': 11}
R4 GS2 type counts at plotted ky (gamma>0): {'KBM': 14, 'MTM': 13, 'mixed': 8}


## Plot

One figure per (case, code): four ky panels, γ vs β (GS2's β number), with TGLF V5 in every figure for comparison.
Lines follow the dominant mode (argmax γ); `show_subdominant` adds the other mode (dashed, open markers).
Markers: ▲ MTM, ■ KBM, ● mixed.

In [7]:
show_subdominant = False   # also draw each code's other mode (faint line, open markers)
comparison = "TGLF V5"     # drawn in every figure next to the code
marker = {"MTM": "^", "KBM": "s", "mixed": "o"}
line_style = {"GFTM": ":", "TGLF": "--"}   # by code family, from the label prefix
# Each figure is drawn for gamma and for omega; dominant mode and markers (gamma > gamma_floor) come from gamma
plot_fields = {"growth_rate": r"$\gamma$ [$c_s/a$]", "mode_frequency": r"$\omega$ [$c_s/a$]"}
figs = {}
for f, c in [(f, c) for f in plot_fields for c in cases]:   # one figure per field
    ref, beta = gs2[c], gs2[c].beta.values
    jb = np.arange(beta.size)
    for lab in codes:
        series = [(lab, "C0")] + ([(comparison, "C1")] if lab != comparison else [])
        fig, axes = plt.subplots(1, len(ky_targets), figsize=(4 * len(ky_targets), 3.6), sharey=True)
        for ax, i in zip(axes, ky_index[c]):
            ax.plot(beta, ref[f].values[i], alpha=0.6, color="k", marker="none")
            for j, g in enumerate(ref[f].values[i]):
                if (c, i, j) in gs2_label:
                    ax.plot(beta[j], g, color="k", marker=marker[gs2_label[(c, i, j)]], ls="")
            for name, col in series:
                g, y, d, typ = model[(c, name)].growth_rate.values[i], model[(c, name)][f].values[i], dominant_mode[(c, name)][i], code_label[(c, name)][i]
                ls = line_style[name.split()[0]]
                ax.plot(beta, y[jb, d], color=col, ls=ls, alpha=0.6, marker="none")
                if show_subdominant:
                    ax.plot(beta, y[jb, 1 - d], color=col, ls=ls, alpha=0.4, marker="none")   # NMODES = 2
                for j, m in np.argwhere(g > gamma_floor):
                    if m == d[j] or show_subdominant:
                        ax.plot(beta[j], y[j, m], color=col, marker=marker[typ[j, m]], ls="", mfc=col if m == d[j] else "none")
            ax.set(xlabel=r"$\beta_e$", title=rf"$k_y$ = {float(ref.ky[i]):.3f}")
        key = [Line2D([], [], color="k", ls="", marker="o", label="GS2"),
               *[Line2D([], [], color=col, ls="", marker="o", label=name) for name, col in series],
               *([Line2D([], [], color="grey", alpha=0.4, label="other mode")] if show_subdominant else []),
               *[Line2D([], [], color="k", ls="", marker=m, label=t if t.isupper() else t.capitalize()) for t, m in marker.items()]]
        axes[0].set_ylabel(plot_fields[f])
        axes[-1].legend(handles=key, loc="best", fontsize=8)
        fig.suptitle(rf"{c}", y=1.04)
        figs[(c, lab, f)] = fig
print(len(figs), "figures")
plt.show()


21 figures


/scratch_local/ipykernel_4158049/707744643.py:11: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(1, len(ky_targets), figsize=(4 * len(ky_targets), 3.6), sharey=True)
/scratch_local/ipykernel_4158049/707744643.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
ky_comparisons = {"TGLF default": "C1", "TGLF V3": "C2"}   # all drawn in every single-ky figure
ky_codes = [lab for lab in codes if lab != "TGLF V5"]   # TGLF V5 left out of the single-ky figures
legend_name = {"GFTM w0.4": "GFTM"}   # display name in the single-ky legends; other labels unchanged
reference_beta = {
    "M1_A1_NE": 0.036,
    "R3": 0.094,
    "R4": 0.053,
}
figs_ky = {}   # one figure per (case, code, ky, field); same lines and markers as the panel figures above
for f, c in [(f, c) for f in plot_fields for c in cases]:   # one figure per field
    ref, beta = gs2[c], gs2[c].beta.values
    jb = np.arange(beta.size)
    for lab in ky_codes:
        series = ([] if lab in ky_comparisons else [(lab, "C0")]) + list(ky_comparisons.items())
        key = [Line2D([], [], color="k", ls="", marker="o", label="GS2"),
               *[Line2D([], [], color=col, ls="", marker="o", label=legend_name.get(name, name)) for name, col in series],
               *([Line2D([], [], color="grey", alpha=0.4, label="other mode")] if show_subdominant else []),
               *[Line2D([], [], color="k", ls="", marker=m, label=t if t.isupper() else t.capitalize()) for t, m in marker.items()]]
        for i in ky_index[c]:
            fig, ax = plt.subplots(figsize=(5, 3.8))
            ax.plot(beta, ref[f].values[i], alpha=0.6, color="k", marker="none")
            for j, g in enumerate(ref[f].values[i]):
                if (c, i, j) in gs2_label:
                    ax.plot(beta[j], g, alpha=0.6, color="k", marker=marker[gs2_label[(c, i, j)]], ls="")
            for name, col in series:
                g, y, d, typ = model[(c, name)].growth_rate.values[i], model[(c, name)][f].values[i], dominant_mode[(c, name)][i], code_label[(c, name)][i]
                ls = line_style[name.split()[0]]
                ax.plot(beta, y[jb, d], alpha=0.6, color=col, ls=ls, marker="none")
                if show_subdominant:
                    ax.plot(beta, y[jb, 1 - d], color=col, ls=ls, alpha=0.4, marker="none")   # NMODES = 2
                for j, m in np.argwhere(g > gamma_floor):
                    if m == d[j] or show_subdominant:
                        ax.plot(beta[j], y[j, m], alpha=0.6, color=col, marker=marker[typ[j, m]], ls="", mfc=col if m == d[j] else "none")

            ky = float(ref.ky[i])
            ax.set(xlabel=r"$\beta_e$", ylabel=plot_fields[f], title=rf"{c}, $k_y$ = {ky:.3f}")
            ax.axvline(reference_beta[c], color="red", linestyle="--", alpha=0.4, marker="none")
            ax.legend(handles=key, loc="best", fontsize=8)
            figs_ky[(c, lab, ky, f)] = fig
            plt.close(fig)   # too many to display, check the saved PNGs
print(len(figs_ky), "figures")
plt.show()


72 figures


/scratch_local/ipykernel_4158049/2433615849.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Mean absolute error against GS2 (poster table)

Per case, code and field: mean |code − GS2| over every (ky, β) cell of GS2's grid, code on its dominant mode
(as plotted). Cells with GS2 `growth_rate_tolerance` ≥ `TOL_MAX` and non-finite cells are dropped, as in
`Linear_Plotting.ipynb`. Written to `Results/FuseNet_Poster/<analysis_name>/` as CSV (long form, with N),
plain text and a LaTeX tabular (lowest MAE per row highlighted) that is pasted into the poster.


In [ ]:
# Mean absolute error against GS2 over every (ky, beta) cell, dominant mode as in the plots. Same definition as
# Linear_Plotting.ipynb: cells with GS2 growth_rate_tolerance >= TOL_MAX are dropped, as are non-finite cells.
import pandas as pd

TOL_MAX = 0.3
case_name = {"M1_A1_NE": "MAST 48567"}   # display names; others unchanged
field_sym = {"growth_rate": r"$\gamma$", "mode_frequency": r"$\omega$"}

rows = []
for c in cases:
    keep = np.ones(gs2[c].growth_rate.shape, bool)
    if "growth_rate_tolerance" in gs2[c]:
        keep = gs2[c].growth_rate_tolerance.transpose("ky", "beta").values < TOL_MAX
    for lab in codes:
        d = dominant_mode[(c, lab)][..., None]   # (ky, beta, 1)
        for f in field_sym:
            g = np.take_along_axis(model[(c, lab)][f].transpose("ky", "beta", "mode").values, d, -1)[..., 0]
            err = np.abs(g - gs2[c][f].transpose("ky", "beta").values)[keep]
            err = err[np.isfinite(err)]
            rows.append({"case": c, "field": f, "code": lab, "N": err.size, "MAE": err.mean()})
mae = pd.DataFrame(rows)
mae_table = mae.pivot(index=["case", "field"], columns="code", values="MAE").reindex(
    index=pd.MultiIndex.from_product([cases, list(field_sym)]), columns=list(codes))
print(mae_table.to_string(float_format="{:.3f}".format))

# LaTeX in the poster's table style: codes grouped by family, lowest MAE per row highlighted.
families = {}
for lab in codes:
    families.setdefault(lab.split()[0], []).append(lab)
head = " & & " + " & ".join(rf"\multicolumn{{{len(v)}}}{{c}}{{{k}}}" for k, v in families.items()) + r" \\"
rules, col = [], 3
for v in families.values():
    rules.append(rf"\cmidrule(lr){{{col}-{col + len(v) - 1}}}")
    col += len(v)
sub = "Case & Field & " + " & ".join(lab.split(maxsplit=1)[1].replace("default", "Default") for lab in codes) + r" \\"
lines = [rf"\begin{{tabular}}{{ll{'r' * len(codes)}}}", r"\toprule", head, " ".join(rules), sub]
for c in cases:
    lines.append(r"\midrule")
    for k, f in enumerate(field_sym):
        vals = mae_table.loc[(c, f)].values
        row = [(r"\cellcolor{green!30}" if v == np.nanmin(vals) else "") + f"{v:.3f}" for v in vals]
        first = rf"\multirow{{2}}{{*}}{{{case_name.get(c, c)}}}" if k == 0 else ""
        lines.append(f"{first} & {field_sym[f]} & " + " & ".join(row) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}"]

results_dir = ROOT / "Results" / "FuseNet_Poster" / analysis_name
results_dir.mkdir(parents=True, exist_ok=True)
mae.to_csv(results_dir / "mae_vs_gs2.csv", index=False)
(results_dir / "mae_vs_gs2.txt").write_text(mae_table.to_string(float_format="{:.3f}".format) + "\n")
(results_dir / "mae_vs_gs2.tex").write_text("\n".join(lines) + "\n")
print("Saved MAE table to", results_dir)

## Save

In [9]:
output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
suffix = {"growth_rate": "", "mode_frequency": "_omega"}

print("saved", len(figs), "panel and", len(figs_ky), "single-ky figures to", output_dir)

## Interpretation

Left to the reader. Code-side markers use T and ω only (no flux weights), GS2 markers use all four indicators, so a
GS2 "mixed" next to a code "KBM" can be a difference in the test rather than in the physics.

## R4 at high β: why GFTM predicts a strong KBM where GS2 is near marginal (Fusion_PhD-38e9.9)

Above β ≈ 0.07 (GS2 number) GS2's dominant R4 mode at ky ≈ 0.1–0.3 is near marginal, while every GFTM width
predicts a growing ballooning-parity mode, more strongly the larger WIDTH is. This section compares the two at the
level of the mode itself. Every leaf is a `PyroScan` loaded from its `pyroscan.nc`, with pyro's `Extent` (95% of
∫|field|²dθ) added to its `gk_output`. Per-run `Pyro` reads remain only for GS2 γ(t) (not stored in a scan) and the
input decks of the β′ and ideal-ballooning checks. Dominant mode = `argmax` of γ over `mode`.

Leaves: GFTM `scan_beta_f05/R4/w0p{4,6,7,8}` (FILTER 0.5) and `scan_beta/R4/parameter_scan_gftm_default`
(WIDTH 1.74, otherwise the same deck: FIND_WIDTH=F, FILTER 0.5, NBASIS_MAX 12, NXGRID 32, NMODES 2, USE_BPAR=T).
The 1.74 leaf labels β with GS2's number and ky to 3 decimals; it is matched by index (asserted below).
GS2 convergence: `growth_rate_tolerance` from GS2's `pyroscan.nc`, thresholds from Fusion_PhD-6vvq
(< 1e-2 converged, ≥ 1e-1 unconverged; unconverged GS2 points are drawn open).

In [11]:
from pyrokinetics.diagnostics.extent import Extent
from matplotlib.ticker import StrMethodFormatter
import pandas as pd

r4 = "R4"
r4_codes = {   # label -> (code, project, leaf, WIDTH)
    "GFTM w0.4": ("GFTM", "scan_beta_f05", "w0p4", 0.4),
    "GFTM w0.6": ("GFTM", "scan_beta_f05", "w0p6", 0.6),
    "GFTM w0.7": ("GFTM", "scan_beta_f05", "w0p7", 0.7),
    "GFTM w0.8": ("GFTM", "scan_beta_f05", "w0p8", 0.8),
    "GFTM w1.74": ("GFTM", "scan_beta", "parameter_scan_gftm_default", 1.74),
}
r4_eig = ["GFTM w0.4", "GFTM w0.8", "GFTM w1.74"]   # widths drawn in the eigenfunction figures
r4_ky = [0.1, 0.2, 0.3]                            # snapped to GS2's grid
beta_high, beta_low = 0.15, 0.02                   # GS2 beta numbers: the high-beta cells and the control
tol_unconverged = 1e-1
fraction = 0.95
fields = ["phi", "apar", "bpar"]
field_label = {"phi": r"$\phi$", "apar": r"$A_\parallel$", "bpar": r"$B_\parallel$"}
colour = {lab: plt.cm.viridis(x) for lab, x in zip(r4_codes, np.linspace(0, 0.9, len(r4_codes)))}


def v(da):
    return np.asarray(getattr(da.data, "magnitude", da.data))


def load_r4(code, project, leaf):
    s = scan(code, project, r4, leaf)
    Extent(s.gk_output, fraction=fraction)
    return s


r4_pyroscan = {"GS2": load_r4(gs2_code, gs2_project, gs2_scan)}
r4_pyroscan |= {lab: load_r4(*spec[:3]) for lab, spec in r4_codes.items()}
R = {lab: s.gk_output.data for lab, s in r4_pyroscan.items()}
for lab, d in R.items():
    assert np.allclose(v(d.ky), v(R["GS2"].ky), atol=1e-3) and d.beta.size == R["GS2"].beta.size, lab


def dominant(d, name):
    "variable of the dominant (argmax gamma over mode) mode; GS2 has one mode"
    if "mode" not in d[name].dims:
        return d[name]
    m = np.nan_to_num(v(d.growth_rate), nan=-np.inf).argmax(-1)
    return d[name].isel(mode=xr.DataArray(m, dims=("ky", "beta")))


beta_r4 = gs2[r4].beta.values            # GS2's numbers, the x axis throughout
iky = [int(np.abs(gs2[r4].ky.values - k).argmin()) for k in r4_ky]
jhi, jlo = (int(np.abs(beta_r4 - b).argmin()) for b in (beta_high, beta_low))
tol = v(R["GS2"].growth_rate_tolerance)
print("ky:", [round(float(gs2[r4].ky[i]), 3) for i in iky], "| beta high", beta_r4[jhi], "control", beta_r4[jlo])

ky: [0.112, 0.2, 0.3] | beta high 0.15 control 0.02


### Where GFTM over-predicts

Dominant γ per code at the plotted ky for β ≥ 0.07, with GS2's tolerance. The eigenfunction cells are the high-β
cells at ky ≈ 0.11 and 0.2 and the low-β control at ky ≈ 0.2, where the codes agree.

In [12]:
g = {lab: v(dominant(d, "growth_rate")) for lab, d in R.items()}
rows = [{"ky": round(float(gs2[r4].ky[i]), 3), "beta": beta_r4[j], "GS2 tol": tol[i, j],
         **{lab: g[lab][i, j] for lab in R}} for i in iky for j in range(beta_r4.size) if beta_r4[j] >= 0.07]
with pd.option_context("display.float_format", "{:.3g}".format, "display.width", 200):
    print(pd.DataFrame(rows).to_string(index=False))

cells = {"control": (iky[1], jlo), "high, ky 0.11": (iky[0], jhi), "high, ky 0.2": (iky[1], jhi)}
for name, (i, j) in cells.items():
    print(f"{name}: ky {float(gs2[r4].ky[i]):.3f} beta {beta_r4[j]:.2f}  " + "  ".join(f"{lab} {g[lab][i, j]:+.3f}" for lab in R))

   ky  beta  GS2 tol      GS2  GFTM w0.4  GFTM w0.6  GFTM w0.7  GFTM w0.8  GFTM w1.74
0.112  0.07     1.06 -0.00781     0.0284     0.0532     0.0934      0.104       0.579
0.112  0.08      1.1 -0.00202     0.0264     0.0864      0.116      0.226       0.642
0.112  0.09     1.06 -0.00682     0.0283      0.104      0.184      0.301         0.7
0.112   0.1      1.9 -0.00263     0.0281      0.117      0.255      0.391       0.752
0.112  0.11     1.09 -0.00191     0.0494      0.156      0.334       0.49       0.798
0.112  0.12     1.19  -0.0057     0.0548      0.222      0.428      0.584       0.837
0.112  0.13     1.09 -0.00325     0.0397      0.294      0.521      0.665        0.87
0.112  0.14     1.23  0.00263     0.0434       0.39      0.606      0.735       0.898
0.112  0.15     1.57  0.00748     0.0407       0.48       0.68      0.793       0.921
0.112  0.16     1.91   0.0105     0.0937      0.561      0.745      0.842        0.94
0.112  0.17     3.08    0.013      0.127      0.634   

### (1) Eigenfunctions

Dominant mode of each code. Every field is divided by the complex φ at |φ|'s peak, so φ is real and 1 there and
A∥, B∥ keep their amplitude and phase relative to φ. Phase is drawn only where |field| > 1% of its own peak.

In [13]:
def eig(d, i, j):
    e = dominant(d, "eigenfunctions").isel(ky=i, beta=j)
    f = {k: v(e.sel(field=k)) for k in fields}
    p = f["phi"]
    norm = p[np.nanargmax(np.abs(p))]
    return v(e.theta), {k: x / norm for k, x in f.items()}


r4_eig_codes = ["GS2", *r4_eig]
line_colour = {"GS2": "k", **colour}
fig_eig, axes = plt.subplots(len(cells), len(fields), figsize=(15, 3.6 * len(cells)), sharex=True)
fig_ph, axes_ph = plt.subplots(len(cells), 2, figsize=(11, 3.6 * len(cells)), sharex=True)
for row, (name, (i, j)) in enumerate(cells.items()):
    for lab in r4_eig_codes:
        th, f = eig(R[lab], i, j)
        for col, k in enumerate(fields):
            axes[row, col].plot(th, np.abs(f[k]), color=line_colour[lab], label=lab, marker="none")
        for col, k in enumerate(("phi", "apar")):
            a = np.abs(f[k])
            ok = a > 0.01 * np.nanmax(a)
            axes_ph[row, col].plot(th[ok], np.angle(f[k][ok]), color=line_colour[lab], ls="", marker=".", ms=3)
    title = rf"$k_y$ = {float(gs2[r4].ky[i]):.2f}, $\beta$ = {beta_r4[j]:.2f}"
    for col, k in enumerate(fields):
        axes[row, col].set_title(f"{title}: |{field_label[k]}|", fontsize=16)
    for col, k in enumerate(("phi", "apar")):
        axes_ph[row, col].set_title(f"{title}: arg {field_label[k]}", fontsize=16)
for ax in [*axes.flat, *axes_ph.flat]:
    ax.tick_params(labelsize=14)
for ax in [*axes[-1], *axes_ph[-1]]:
    ax.set_xlabel(r"$\theta$", fontsize=16)
axes[0, 0].legend(fontsize=12)
axes_ph[0, 0].legend(handles=[Line2D([], [], color=line_colour[l], label=l) for l in r4_eig_codes], fontsize=12)
fig_eig.tight_layout()
fig_ph.tight_layout()
plt.show()

/scratch_local/ipykernel_4158049/2899336431.py:33: UserWarning: The figure layout has changed to tight
  fig_eig.tight_layout()
/scratch_local/ipykernel_4158049/2899336431.py:34: UserWarning: The figure layout has changed to tight
  fig_ph.tight_layout()
/scratch_local/ipykernel_4158049/2899336431.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### (2) Extents vs β

pyro `Extent` (θ range holding 95% of ∫|field|²dθ) of the dominant mode. Top: the extents themselves (grey dotted =
GS2's θ box, 2 × 34.6 rad; GFTM's wavefunction grid is ±28.3 rad). Bottom: log(Extent Ratio) = log₂(GFTM / GS2).

In [14]:
ext = {lab: {k: v(dominant(d, "extent").sel(field=k)) for k in ("phi", "apar")} for lab, d in R.items()}
box = float(np.ptp(v(R["GS2"].theta)))
unconv = tol >= tol_unconverged

fig_ext, axes = plt.subplots(2, len(iky), figsize=(5 * len(iky), 8), sharex=True)
fig_rat, axes_r = plt.subplots(2, len(iky), figsize=(5 * len(iky), 8), sharex=True, sharey=True)
for col, i in enumerate(iky):
    for row, k in enumerate(("phi", "apar")):
        ax, axr = axes[row, col], axes_r[row, col]
        e2 = ext["GS2"][k][i]
        ax.plot(beta_r4, e2, color="k", label="GS2", marker="none")
        ax.plot(beta_r4[~unconv[i]], e2[~unconv[i]], color="k", ls="", marker="o", ms=5)
        ax.plot(beta_r4[unconv[i]], e2[unconv[i]], color="k", ls="", marker="o", ms=5, mfc="none")
        for lab in r4_codes:
            ax.plot(beta_r4, ext[lab][k][i], color=colour[lab], label=lab, marker="none")
            axr.plot(beta_r4, np.log2(ext[lab][k][i] / e2), color=colour[lab], label=lab, marker="none")
        ax.axhline(box, color="0.5", ls=":", lw=1, marker="")
        axr.axhline(0, color="0.3", lw=1, marker="")
        ax.set_yscale("log")
        ax.set_title(rf"{field_label[k]}, $k_y$ = {float(gs2[r4].ky[i]):.2f}", fontsize=16)
        axr.set_title(rf"{field_label[k]}, $k_y$ = {float(gs2[r4].ky[i]):.2f}", fontsize=16)
for a in (axes, axes_r):
    for ax in a.flat:
        ax.tick_params(labelsize=14)
    for ax in a[-1]:
        ax.set_xlabel(r"GS2 $\beta$", fontsize=16)
for row, k in enumerate(("phi", "apar")):
    axes[row, 0].set_ylabel(f"{field_label[k]} extent [rad]", fontsize=16)
    axes_r[row, 0].set_ylabel("log(Extent Ratio)", fontsize=16)
axes[0, 0].legend(fontsize=12)
axes_r[0, 0].legend(fontsize=12)
fig_ext.tight_layout()
fig_rat.tight_layout()
plt.show()

/scratch_local/ipykernel_4158049/488973952.py:32: UserWarning: The figure layout has changed to tight
  fig_ext.tight_layout()
/scratch_local/ipykernel_4158049/488973952.py:33: UserWarning: The figure layout has changed to tight
  fig_rat.tight_layout()
/scratch_local/ipykernel_4158049/488973952.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### (3) γ, ω, T and mode type vs β

Dominant mode per code. GS2 indicators from each run (`mode_classification.indicators` + `classify`, full
criteria); code type from T and ω only (`code_type` above). Current cutoffs are printed. Markers where γ > 0:
▲ MTM, ■ KBM, ● mixed; GS2 open = unconverged (tol ≥ 0.1). TGLF default / V5 in grey for context (no markers).
GFTM WIDTH 1.74 has no stored `tearing_parameter`, so its T is computed with pyro's
`FieldLine.compute_linear_tearing_parameter` (the function that wrote the stored values for the other leaves) from
that leaf's stored A∥ eigenfunction (`eig_T`, checked against the stored T of every other leaf above).

In [15]:
print("cutoffs: T_BAL", mc.T_BAL, "T_TEAR", mc.T_TEAR, "CHI_KBM", mc.CHI_KBM)


def gs2_run(i, j):
    return data_root / gs2_code / run_template / gs2_project / r4 / gs2_scan / f"ky{float(gs2[r4].ky[i]):.3f}_beta{beta_r4[j]:.3f}"


gs2_ind = {(i, j): gs2_indicators(r4, i, j) for i in iky for j in range(beta_r4.size)}
R["GFTM w1.74"]["tearing_parameter"] = (("ky", "beta", "mode"), eig_T(r4_pyroscan["GFTM w1.74"]))

w = {lab: v(dominant(d, "mode_frequency")) for lab, d in R.items()}
T = {lab: v(dominant(d, "tearing_parameter")) for lab, d in R.items() if lab != "GS2"}
T["GS2"] = np.full(g["GS2"].shape, np.nan)
for (i, j), ind in gs2_ind.items():
    T["GS2"][i, j] = ind["T"]
label = {lab: code_type(T[lab], w[lab]) for lab in r4_codes}
label["GS2"] = {k: str(mc.classify(ind)) for k, ind in gs2_ind.items()}
tglf = {lab: model[(r4, lab)] for lab in ("TGLF default", "TGLF V5")}
tglf_dom = {lab: ds.growth_rate.fillna(-np.inf).argmax("mode") for lab, ds in tglf.items()}

fig_gwt, axes = plt.subplots(3, len(iky), figsize=(5 * len(iky), 11), sharex=True)
for col, i in enumerate(iky):
    for lab, ls in zip(tglf, ("--", ":")):
        ds, m = tglf[lab], tglf_dom[lab]
        for row, var in enumerate(("growth_rate", "mode_frequency", "tearing_parameter")):
            axes[row, col].plot(beta_r4, ds[var].isel(mode=m).values[i], color="0.6", ls=ls, label=lab, marker="none")
    for lab in ["GS2", *r4_codes]:
        c = line_colour[lab]
        for row, y in enumerate((g[lab][i], w[lab][i], T[lab][i])):
            axes[row, col].plot(beta_r4, y, color=c, label=lab, marker="none")
        for j in range(beta_r4.size):
            if g[lab][i, j] > gamma_floor:
                t = label[lab][i, j]
                open_ = lab == "GS2" and unconv[i, j]
                axes[0, col].plot(beta_r4[j], g[lab][i, j], color=c, ls="", marker=marker[t], ms=5, mfc="none" if open_ else c)
    axes[0, col].set_title(rf"$k_y$ = {float(gs2[r4].ky[i]):.2f}", fontsize=16)
    axes[2, col].set_yscale("log")
    axes[2, col].axhline(mc.T_BAL, color="0.3", ls=":", lw=1, marker="")
    axes[2, col].axhline(mc.T_TEAR, color="0.3", ls=":", lw=1, marker="")
    axes[1, col].axhline(0, color="0.3", lw=1, marker="")
for row, yl in enumerate((r"$\gamma$ [$c_s/a$]", r"$\omega$ [$c_s/a$]", "tearing parameter T")):
    axes[row, 0].set_ylabel(yl, fontsize=16)
for ax in axes.flat:
    ax.tick_params(labelsize=14)
for ax in axes[-1]:
    ax.set_xlabel(r"GS2 $\beta$", fontsize=16)
axes[0, 0].legend(fontsize=10, ncol=2)
fig_gwt.tight_layout()
plt.show()

for i in iky:
    for lab in ["GS2", *r4_codes]:
        print(f"ky {float(gs2[r4].ky[i]):.3f} {lab:11s}", " ".join(
            f"{label[lab][i, j][0] if g[lab][i, j] > 0 else '-'}" for j in range(beta_r4.size)))

cutoffs: T_BAL 0.05 T_TEAR 0.15 CHI_KBM (0.25, 4.0)


ky 0.112 GS2         m m m m m m - - - - - - - M M M M M M
ky 0.112 GFTM w0.4   m m m m m K K K K K K K K K K K K K K
ky 0.112 GFTM w0.6   m m m m m K K K K K K K K K K K K K K
ky 0.112 GFTM w0.7   m m m m K K K K K K K K K K K K K K K
ky 0.112 GFTM w0.8   m m m m K K K K K K K K K K K K K K K
ky 0.112 GFTM w1.74  m m m K K K K K K K K K K K K K K K K
ky 0.200 GS2         m K K K K K - - - - - - - - - - - M M
ky 0.200 GFTM w0.4   m m K K K K K K K K K K K K K K K K K
ky 0.200 GFTM w0.6   m m K K K K K K K K K K K K K K K K K
ky 0.200 GFTM w0.7   m m K K K K K K K K K K K K K K K K K
ky 0.200 GFTM w0.8   m m K K K K K K K K K K K K K K K K K
ky 0.200 GFTM w1.74  m K K K K K K K K K K K K K K K K m m
ky 0.300 GS2         m K K K K - - - - - - - - - - - - - M
ky 0.300 GFTM w0.4   m K K K K K K K K K K K K K K K K K K
ky 0.300 GFTM w0.6   m K K K K K K K K K K K K K K K K K K
ky 0.300 GFTM w0.7   m K K K K K K K K K K K K K K K K K K
ky 0.300 GFTM w0.8   m K K K K K K K K K K K K K K K K K

/scratch_local/ipykernel_4158049/3677701063.py:48: UserWarning: The figure layout has changed to tight
  fig_gwt.tight_layout()
/scratch_local/ipykernel_4158049/3677701063.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### (7) GFTM convergence study (follow-ups 2-4, user go-ahead 2026-10-05)
Leaves under `GFTM/Runs/scan_beta_f05_conv/R4/` (Fusion_PhD `results/r4_gftm_convergence/config.yaml`), each the
`scan_beta_f05/R4/w0p8` deck with one setting changed, at the three plotted ky: `width_0p2/0p25/0p3/0p5` (WIDTH, 19 β),
`nb<NBASIS_MAX>_nx<NXGRID>_w0p8/w1p74` (4 β: the β 0.02 control and β 0.10/0.15/0.19), and `nobpar_w0p4/0p8/1p74`
(`numerics.bpar = False`, i.e. `USE_BPAR = F`, 19 β). Dominant mode = argmax γ. β is GS2's number throughout.

In [21]:
def load_conv(leaf):
    d = load_r4("GFTM", "scan_beta_f05_conv", leaf).gk_output.data
    return d.squeeze("bpar", drop=True) if "bpar" in d.dims else d


C = {leaf: load_conv(leaf) for leaf in
     ["width_0p2", "width_0p25", "width_0p3", "width_0p5", "nobpar_w0p4", "nobpar_w0p8", "nobpar_w1p74",
      *(f"nb{nb}_nx{nx}_w{w}" for w in ("0p8", "1p74") for nb, nx in ((12, 32), (16, 32), (20, 32), (12, 48), (12, 64), (20, 64)))]}
jconv = [1, 9, 14, 18]   # beta indices of the nb*_nx* leaves: GS2 beta 0.02, 0.10, 0.15, 0.19
for leaf, d in C.items():
    assert np.allclose(v(d.ky), v(R["GS2"].ky)[iky], atol=1e-3), leaf
    assert d.beta.size == (4 if leaf.startswith("nb") else beta_r4.size), leaf
gc = {leaf: v(dominant(d, "growth_rate")) for leaf, d in C.items()}
gk = {w: gc[f"width_{str(w).replace('.', 'p')}"] for w in (0.2, 0.25, 0.3, 0.5)}
gk |= {spec[3]: g[lab][iky] for lab, spec in r4_codes.items()}
gk = dict(sorted(gk.items()))
wcol = {w: plt.cm.viridis(x) for w, x in zip(gk, np.linspace(0, 0.92, len(gk)))}
gs2_hi = beta_r4 >= 0.07

fig_ws, axes = plt.subplots(1, len(iky), figsize=(5 * len(iky), 4.8), sharey=True)
for col, i in enumerate(iky):
    ax = axes[col]
    ax.plot(beta_r4, g["GS2"][i], color="k", lw=2.5, label="GS2", marker="none")
    ax.plot(beta_r4[unconv[i]], g["GS2"][i, unconv[i]], color="k", ls="", marker="o", ms=5, mfc="none")
    for w, y in gk.items():
        ax.plot(beta_r4, y[col], color=wcol[w], label=f"W {w:g}", marker="none")
    ax.axhline(0, color="0.3", lw=1, marker="")
    ax.set_title(rf"R4, $k_y$ = {float(gs2[r4].ky[i]):.2f}", fontsize=16)
    ax.set_xlabel(r"GS2 $\beta$", fontsize=16)
    ax.tick_params(labelsize=14)
axes[0].set_ylabel(r"$\gamma$ [$c_s/a$]", fontsize=16)
axes[-1].legend(fontsize=11, ncol=2)
fig_ws.tight_layout()
plt.show()
for col, i in enumerate(iky):
    print(f"ky {float(gs2[r4].ky[i]):.3f}: max gamma at beta>=0.07 (GS2 {g['GS2'][i, gs2_hi].max():+.3f}) | " +
          "  ".join(f"W{w:g} {y[col, gs2_hi].max():.3f}" for w, y in gk.items()))

ky 0.112: max gamma at beta>=0.07 (GS2 +0.018) | W0.2 0.047  W0.25 0.033  W0.3 0.058  W0.4 0.198  W0.5 0.506  W0.6 0.753  W0.7 0.886  W0.8 0.948  W1.74 0.975
ky 0.200: max gamma at beta>=0.07 (GS2 +0.012) | W0.2 0.072  W0.25 0.121  W0.3 0.125  W0.4 0.262  W0.5 0.438  W0.6 0.635  W0.7 0.729  W0.8 0.778  W1.74 0.737
ky 0.300: max gamma at beta>=0.07 (GS2 +0.005) | W0.2 0.088  W0.25 0.107  W0.3 0.101  W0.4 0.225  W0.5 0.232  W0.6 0.243  W0.7 0.283  W0.8 0.325  W1.74 0.347


/scratch_local/ipykernel_4158049/3216528222.py:33: UserWarning: The figure layout has changed to tight
  fig_ws.tight_layout()
/scratch_local/ipykernel_4158049/3216528222.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
res = [(12, 32), (16, 32), (20, 32), (12, 48), (12, 64), (20, 64)]
fig_res, axes = plt.subplots(2, len(iky), figsize=(5 * len(iky), 8), sharex=True, sharey="row")
for row, w in enumerate(("0p8", "1p74")):
    for col, i in enumerate(iky):
        ax = axes[row, col]
        for b, (k, j) in enumerate(zip(range(4), jconv)):
            y = [gc[f"nb{nb}_nx{nx}_w{w}"][col, k] for nb, nx in res]
            ax.plot(range(len(res)), y, color=f"C{b}", marker="o", ms=5, label=rf"$\beta$ = {beta_r4[j]:.2f}")
            ax.axhline(g["GS2"][i, j], color=f"C{b}", ls=":", marker="")
        ax.set_title(rf"WIDTH {w.replace('p', '.')}, $k_y$ = {float(gs2[r4].ky[i]):.2f}", fontsize=16)
        ax.tick_params(labelsize=13)
for ax in axes[-1]:
    ax.set_xticks(range(len(res)), [f"{nb}/{nx}" for nb, nx in res], rotation=45)
    ax.set_xlabel("NBASIS_MAX / NXGRID", fontsize=16)
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\gamma$ [$c_s/a$]", fontsize=16)
axes[0, 0].legend(fontsize=11)
fig_res.tight_layout()
plt.show()
for w in ("0p8", "1p74"):
    base = gc[f"nb12_nx32_w{w}"]
    spread = np.max([np.abs(gc[f"nb{nb}_nx{nx}_w{w}"] - base) for nb, nx in res], axis=0)
    print(f"W{w}: max |gamma - gamma(12/32)| over settings, per ky x beta {beta_r4[jconv].round(2)}:\n", spread.round(3))
    for col, i in enumerate(iky):
        print(f"  ky {float(gs2[r4].ky[i]):.3f}: " + "  ".join(
            f"{nb}/{nx} " + "/".join(f"{x:.3f}" for x in gc[f'nb{nb}_nx{nx}_w{w}'][col]) for nb, nx in res))

W0p8: max |gamma - gamma(12/32)| over settings, per ky x beta [0.02 0.1  0.15 0.19]:
 [[0.017 0.312 0.376 0.269]
 [0.011 0.163 0.278 0.241]
 [0.001 0.123 0.209 0.112]]
  ky 0.112: 12/32 0.193/0.391/0.793/0.948  16/32 0.204/0.299/0.739/0.933  20/32 0.209/0.216/0.643/0.877  12/48 0.193/0.282/0.651/0.825  12/64 0.193/0.226/0.546/0.730  20/64 0.210/0.078/0.417/0.679
  ky 0.200: 12/32 0.354/0.357/0.652/0.778  16/32 0.359/0.301/0.640/0.808  20/32 0.364/0.288/0.579/0.782  12/48 0.354/0.257/0.517/0.643  12/64 0.354/0.193/0.429/0.537  20/64 0.365/0.207/0.374/0.582
  ky 0.300: 12/32 0.440/0.284/0.321/0.277  16/32 0.441/0.271/0.381/0.332  20/32 0.441/0.192/0.387/0.348  12/48 0.440/0.206/0.185/0.252  12/64 0.440/0.160/0.112/0.252  20/64 0.441/0.179/0.146/0.165
W1p74: max |gamma - gamma(12/32)| over settings, per ky x beta [0.02 0.1  0.15 0.19]:
 [[0.05  0.259 0.236 0.203]
 [0.041 0.306 0.302 0.284]
 [0.045 0.311 0.56  0.155]]
  ky 0.112: 12/32 0.250/0.752/0.921/0.975  16/32 0.238/0.900/1.052/1.078

/scratch_local/ipykernel_4158049/4066944058.py:18: UserWarning: The figure layout has changed to tight
  fig_res.tight_layout()
/scratch_local/ipykernel_4158049/4066944058.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### (8) Is GFTM's large B∥/φ a normalisation convention? Ratio to GS2 vs ky at the β 0.02 control
GFTM's B∥ eigenfunction is its internal field σ, solved with `betasig = 0.5·betae/ky²` (GFTM `gftm_eigensolver.f90`),
and pyro's GFTM reader applies units only, no ky factor. If σ is δB∥ times a power of ky, the GFTM/GS2 ratio of
peak|B∥|/peak|φ| should follow that power of ky where the two codes find the same mode. All 42 ky at β 0.02, cells with
GS2 converged (tol < 1e-2) and γ > 0 in both. A∥ for comparison. Slope from a straight-line fit in log-log.

In [24]:
def peak_ratio(d, k):
    e = dominant(d, "eigenfunctions")
    return np.nanmax(np.abs(v(e.sel(field=k))), axis=-1) / np.nanmax(np.abs(v(e.sel(field="phi"))), axis=-1)

ky_all = v(R["GS2"].ky)
ok_gs2 = (tol[:, jlo] < 1e-2) & (g["GS2"][:, jlo] > 0)
fig_bn, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
for ax, k in zip(axes, ("bpar", "apar")):
    r2 = peak_ratio(R["GS2"], k)[:, jlo]
    for lab in r4_eig:
        ok = ok_gs2 & (g[lab][:, jlo] > 0)
        y = peak_ratio(R[lab], k)[:, jlo] / r2
        slope = np.polyfit(np.log(ky_all[ok]), np.log(y[ok]), 1)[0]
        ax.plot(ky_all[ok], y[ok], color=colour[lab], marker="o", ms=4, label=f"{lab.replace('GFTM ', 'W ')}: slope {slope:.2f}")
        print(f"{k} {lab}: n {ok.sum()}, ratio GFTM/GS2 at ky 0.11/0.2/0.3 "
              f"{'/'.join(f'{y[i]:.2f}' for i in iky)}, log-log slope {slope:.2f}")
    ax.axhline(1, color="0.3", lw=1, marker="")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$k_y$", fontsize=16)
    ax.set_title(rf"{field_label[k]}: GFTM / GS2, $\beta$ = {beta_r4[jlo]:.2f}", fontsize=16)
    ax.tick_params(labelsize=14)
    ax.legend(fontsize=11)
axes[0].set_ylabel(r"(max|field| / max|$\phi$|) ratio", fontsize=15)
fig_bn.tight_layout()
plt.show()
conv_figs = {"gftm_gamma_vs_nbasis_per_beta_r4": fig_res, 
             }
for name, fig in conv_figs.items():
    fig.savefig(output_dir / f"{name}.png", dpi=150, bbox_inches="tight")
print("saved", len(conv_figs), "figures")